# PSNR

Peak Signal-to-Noise Ratio. Mide la relación entre la señal máxima posible y el ruido introducido por la compresión. Se basa en el MSE entre imagen original y distorsionada. Valores más altos indican mejor calidad (típicamente 30–50 dB para buena calidad).

In [1]:
import numpy as np
from PIL import Image
from skimage.metrics import peak_signal_noise_ratio

def calcular_psnr(img_orig_path, img_comp_path):
    orig = np.array(Image.open(img_orig_path).convert('RGB'))
    comp = np.array(Image.open(img_comp_path).convert('RGB'))
    return peak_signal_noise_ratio(orig, comp, data_range=255)

# Implementación manual:
def psnr_manual(img_orig_path, img_comp_path):
    orig = np.array(Image.open(img_orig_path).convert('RGB'),
                    dtype=np.float64)
    comp = np.array(Image.open(img_comp_path).convert('RGB'),
                    dtype=np.float64)
    mse = np.mean((orig - comp) ** 2)
    if mse == 0:
        return float('inf')
    return 10 * np.log10(255**2 / mse)

# SSIM

Structural Similarity Index. Mide similitud entre dos imágenes comparando luminancia, contraste y estructura. Resultado entre -1 y 1; donde 1 = similitud perfecta. Mejor correlación con percepción humana que PSNR.

In [2]:
import numpy as np
from PIL import Image
from skimage.metrics import structural_similarity

def calcular_ssim(img_orig_path, img_comp_path):
    orig = np.array(Image.open(img_orig_path).convert('RGB'))
    comp = np.array(Image.open(img_comp_path).convert('RGB'))
    score, mapa = structural_similarity(
        orig, comp,
        channel_axis=2,
        data_range=255,
        full=True
    )
    return score, mapa  # score: valor global, mapa: SSIM por pixel

# MS-SSIM

Multi-Scale Structural Similarity. Extensión de SSIM que evalúa similitud estructural en múltiples escalas espaciales (resoluciones). Captura degradaciones visuales tanto en detalles finos como en estructuras globales.

In [4]:
import torch
from pytorch_msssim import ms_ssim
from PIL import Image
import numpy as np

def calcular_ms_ssim(img_orig_path, img_comp_path):
    def load_tensor(path):
        arr = np.array(Image.open(path).convert('RGB'),
                       dtype=np.float32) / 255.0
        t = torch.from_numpy(arr).permute(2, 0, 1).unsqueeze(0)
        return t

    orig = load_tensor(img_orig_path)
    comp = load_tensor(img_comp_path)
    score = ms_ssim(orig, comp, data_range=1.0,
                    size_average=True)
    return score.item()

# Alternativa con scikit-image (escala simple):
from skimage.metrics import structural_similarity
def ms_ssim_approx(img_orig_path, img_comp_path, niveles=3):
    orig = np.array(Image.open(img_orig_path).convert('L'),
                    dtype=np.float64)
    comp = np.array(Image.open(img_comp_path).convert('L'),
                    dtype=np.float64)
    scores = []
    for _ in range(niveles):
        s, _ = structural_similarity(orig, comp,
                                     data_range=255, full=True)
        scores.append(s)
        orig = orig[::2, ::2]
        comp = comp[::2, ::2]
    return np.mean(scores)

# LPIPS


Learned Perceptual Image Patch Similarity. Mide diferencia perceptual usando características extraídas de redes neuronales (VGG/AlexNet). Valores más bajos = más similares. Alta correlación con percepción humana.

In [5]:
import lpips
import torch
from PIL import Image
import numpy as np

def calcular_lpips(img_orig_path, img_comp_path,
                   net='vgg'):
    loss_fn = lpips.LPIPS(net=net)

    def load_tensor(path):
        arr = np.array(Image.open(path).convert('RGB'),
                       dtype=np.float32) / 127.5 - 1.0
        t = torch.from_numpy(arr).permute(2, 0, 1).unsqueeze(0)
        return t

    orig = load_tensor(img_orig_path)
    comp = load_tensor(img_comp_path)

    with torch.no_grad():
        dist = loss_fn(orig, comp)
    return dist.item()  # 0=identicas, mayor=mas diferentes

# VIF

Visual Information Fidelity. Mide cuánta información visual del original se conserva en la imagen distorsionada, basándose en teoría de la información y modelos del sistema visual humano.

In [7]:
import numpy as np
from PIL import Image
from sewar.full_ref import vifp

def calcular_vif(img_orig_path, img_comp_path):
    orig = np.array(Image.open(img_orig_path).convert('L'))
    comp = np.array(Image.open(img_comp_path).convert('L'))
    return vifp(orig, comp)

# Implementación aproximada manual:
def vif_aproximado(img_orig_path, img_comp_path,
                   sigma_nsq=2.0):
    from scipy.ndimage import gaussian_filter
    orig = np.array(Image.open(img_orig_path).convert('L'),
                    dtype=np.float64)
    comp = np.array(Image.open(img_comp_path).convert('L'),
                    dtype=np.float64)
    mu1 = gaussian_filter(orig, sigma=1.5)
    mu2 = gaussian_filter(comp, sigma=1.5)
    sigma1_sq = gaussian_filter(orig**2, sigma=1.5) - mu1**2
    sigma12 = gaussian_filter(orig*comp, sigma=1.5) - mu1*mu2
    sigma1_sq = np.maximum(sigma1_sq, 0)
    g = sigma12 / (sigma1_sq + 1e-10)
    sv_sq = np.var(comp - g * orig)
    num = np.sum(np.log2(1 + g**2 * sigma1_sq /
                         (sv_sq + sigma_nsq)))
    den = np.sum(np.log2(1 + sigma1_sq / sigma_nsq))
    return num / (den + 1e-10)

# FSIM

Feature Similarity Index. Evalúa similitud perceptual usando congruencia de fase (Phase Congruency) y magnitud del gradiente. Muy efectivo para detectar degradaciones en bordes, contornos y texturas.

In [14]:
import numpy as np
from PIL import Image
from scipy.ndimage import sobel

def fsim_aproximado(img_orig_path, img_comp_path, T1=0.85, T2=160):
    orig = np.array(Image.open(img_orig_path).convert('L'), dtype=np.float64)
    comp = np.array(Image.open(img_comp_path).convert('L'), dtype=np.float64)

    def grad_mag(img):
        gx = sobel(img, axis=1)
        gy = sobel(img, axis=0)
        return np.sqrt(gx**2 + gy**2)

    # Gradientes
    G1 = grad_mag(orig)
    G2 = grad_mag(comp)

    # Similaridad de gradiente
    GMS = (2 * G1 * G2 + T2) / (G1**2 + G2**2 + T2)

    # Similaridad de intensidad (tipo SSIM simplificado)
    SL = (2 * orig * comp + T1) / (orig**2 + comp**2 + T1)

    # Combinar
    FSIM_aprox = np.mean(GMS * SL)

    return FSIM_aprox

# GMSD

Gradient Magnitude Similarity Deviation. Compara la magnitud de gradientes entre imagen original y distorsionada. Bajo costo computacional y buena correlación perceptual. Valores más bajos = mejor calidad.

In [15]:
import numpy as np
from PIL import Image
from scipy.ndimage import sobel

def calcular_gmsd(img_orig_path, img_comp_path, C=0.0026):
    orig = np.array(Image.open(img_orig_path).convert('L'),
                    dtype=np.float64) / 255.0
    comp = np.array(Image.open(img_comp_path).convert('L'),
                    dtype=np.float64) / 255.0

    def grad_mag(arr):
        gx = sobel(arr, axis=1)
        gy = sobel(arr, axis=0)
        return np.sqrt(gx**2 + gy**2)

    Gx = grad_mag(orig)
    Gy = grad_mag(comp)

    GMS = (2 * Gx * Gy + C) / (Gx**2 + Gy**2 + C)
    mu_GMS = np.mean(GMS)
    N = GMS.size
    gmsd = np.sqrt((1/N) * np.sum((GMS - mu_GMS)**2))
    return gmsd  # menor = mejor calidad

# MS-SSIM + LPIPS (combinado)

Enfoque híbrido que combina MS-SSIM (similitud estructural multiescala) y LPIPS (diferencia perceptual profunda). Captura simultáneamente distorsiones estructurales globales y diferencias perceptuales de alto nivel.

In [16]:
import torch
import lpips
import numpy as np
from PIL import Image
from pytorch_msssim import ms_ssim

def calcular_ms_ssim_lpips(img_orig_path, img_comp_path,
                            alpha=0.5, beta=0.5):
    loss_fn = lpips.LPIPS(net='vgg')

    arr_o = np.array(Image.open(img_orig_path).convert('RGB'),
                     dtype=np.float32)
    arr_c = np.array(Image.open(img_comp_path).convert('RGB'),
                     dtype=np.float32)

    # Tensor normalizado [0,1] para MS-SSIM
    t_o = torch.from_numpy(arr_o/255.0).permute(2,0,1).unsqueeze(0)
    t_c = torch.from_numpy(arr_c/255.0).permute(2,0,1).unsqueeze(0)
    ms = ms_ssim(t_o, t_c, data_range=1.0,
                 size_average=True).item()

    # Tensor normalizado [-1,1] para LPIPS
    lp_o = torch.from_numpy(arr_o/127.5 - 1).permute(2,0,1).unsqueeze(0)
    lp_c = torch.from_numpy(arr_c/127.5 - 1).permute(2,0,1).unsqueeze(0)
    with torch.no_grad():
        lp = loss_fn(lp_o, lp_c).item()

    # Score combinado (mayor = mejor)
    score = alpha * ms + beta * (1 - lp)
    return {'ms_ssim': ms, 'lpips': lp, 'combinado': score}

# VMAF

Video Multi-Method Assessment Fusion. Desarrollado por Netflix. Combina VIF, detalle y SSIM mediante un modelo SVM entrenado con evaluaciones humanas. Resultado 0–100, donde mayor = mejor calidad. Principalmente para video, también aplicable a imágenes.

In [1]:
import subprocess
import json
import os

def calcular_vmaf_ffmpeg(ref_path, dist_path, log_path="vmaf_log.json"):
    try:
        # Comando FFmpeg con libvmaf
        cmd = [
            'ffmpeg',
            '-i', dist_path,
            '-i', ref_path,
            '-lavfi', f'libvmaf=log_fmt=json:log_path={log_path}',
            '-f', 'null', '-'
        ]

        result = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)

        # Verificar errores
        if result.returncode != 0:
            print("[ERROR FFmpeg]")
            print(result.stderr)
            return None

        # Leer resultado
        if not os.path.exists(log_path):
            print("[ERROR] No se generó el log de VMAF")
            return None

        with open(log_path) as f:
            log = json.load(f)

        return log['pooled_metrics']['vmaf']['mean']

    except Exception as e:
        print(f"[ERROR] {e}")
        return None